## 📑 Core Referrals
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AYBWHg)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
%python
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col, regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace
import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim
import boto3
import os
# spark.conf.set("spark.sql.ansi.enabled", "false")

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

unionRefDate =  spark.sql(f'''
                              SELECT min(BusinessDate) Ref
                              FROM `{env_prefix}-cdh-delivery`.repoint.core_mastertable m
						WHERE BusinessDate <= {run_date!r}
						AND BusinessDate > last_day(add_months({run_date!r},-2))
                              ''').collect()[0]['Ref']
unionRefDate = unionRefDate.strftime('%Y-%m-%d')

##### Schema of the table

In [0]:
table_name = "Core_Referrals"

schema = (
"BusinessDate string,"
"CDI string,"
"PartyTaxRegistrationNumber string,"
"ParentPartyId string,"
"AgreementKey string,"
"ExposureKey string,"
"GeminiAccFlag int,"
"ArrearsStatusENG string,"
"AssetSegment string,"
"Balance string,"
"OnBalanceAmount string,"
"OffBalanceAmount string,"
"AgencyCode string,"
"`2901_ΕΚΣ` date,"
"`2902_ΕΑΚΣ` date,"
"`2903_ΕΕΚΣ` date,"
"`2904_ΑΕΚΣ` date,"
"`2905_ΕΕΑΚΣ` date,"
"`2906_ΑΕΑΚΣ` date,"
"`2907_ΕΔΠ` date,"
"`2908_ΕΑΔΠ` date,"
"`2909_ΕΕΔΠ` date,"
"`2910_ΑΕΔΠ` date,"
"`2911_ΕΕΑΔΠ` date,"
"`2912_ΑΕΑΔΠ` date,"
"`2913_ΕΑΠ` date,"
"`2914_ΕΑΑΠ` date,"
"`2915_ΕΕΑΠ` date,"
"`2916_ΑΕΑΠ` date,"
"`2917_ΕΕΑΑΠ` date,"
"`2927_ΠΔΠ` date,"
"`2919_ΕΚ` date,"
"`2920_ΕΑΚ` date,"
"`2921_ΕΕΚ` date,"
"`2926_ΠΚ` date,"
"`2922_ΑΕΚ` date,"
"`2923_ΕΕΑΚ` date,"
"`2924_ΑΕΑΚ` date,"
"`2925_ΑΕΝΕ` date,"
"`2940_ΠΚΣ` date,"
"`4020_ΠΑΓ` date,"
"`4021_ΑΠΑΓ` date,"
"`2931_ΑνΕΚΣ` date,"
"`2932_ΑνΕΔΠ` date,"
"`2933_ΑνΕΑΠ` date,"
"`2934_ΑνΕΚ` date,"
"`2935_EEKE` date,"
"`EDP_Date` date,"
"`EK_Date` date,"
"`ΠΔ_2950` string,"
"`ΚΣΧ_2951` string,"
"`AG_ΕΚΣ` string,"
"`AG_ΕΔΠ` string,"
"`AG_ΕΑΠ` string,"
"`AG_ΕΚ` string,"
"`ΚΑΤΑ_ΔΠ` date,"
"`ΕΚΔ_ΔΠ` date,"
"`ΕΠΙ_ΔΠ` date,"
"`ΚΑΤΑΣΧ` date,"
"`ΚΑΤΑ_ΑΓΩΓ` date,"
"`ΕΚΔ_ΑΓΩΓ` date,"
"`ΚΟΙΝ_ΑΓΩΓ` date,"
"`ΠΡΟΣΗΜ` date,"
"`AG_ΚΑΤΑ_ΔΠ` string,"
"`AG_ΕΚΔ_ΔΠ` string,"
"`AG_ΕΠΙ_ΔΠ` string,"
"`AG_ΚΑΤΑΣΧ` string,"
"`ΑΓΩΓ_4022_4023` string,"
"`ΑΓΩΓ_Category` string,"
"`ΔΠ_Category` string,"
"`ΚΤΣΧ_Category` string,"
"`ΠΡΟΣΗΜ_Category` string,"
"`ΔΠ_Status` string,"
"`ΔΠ_Status_Date` date,"
"`AgogiStatus` string,"
"`Agogi_Status_Date` date,"
"`KSX_Status` string,"
"`KSX_Status_Date` date,"
"`entoli_TA` date,"
"`entoli_dp` date,"
"`entoli_ksx` date,"
"`entoli_pros` date,"
"`ΑΓΩΓ_CurrentAgency` string,"
"`ΔΠ_CurrentAgency` string,"
"`ΚΤΣΧ_CurrentAgency` string,"
"`ΠΡΟΣ_CurrentAgency` string,"
"`2903_ΕΕΚΣ_Time` bigint,"
"`2904_ΑΕΚΣ_Time` bigint,"
"`2931_ΑνΕΚΣ_Time` bigint,"
"`Ekkr_ΕΚΣ_Time` bigint,"
"`2905_ΕΕΑΚΣ_Time` bigint,"
"`2906_ΑΕΑΚΣ_Time` bigint,"
"`Ekkr_ΕΑΚΣ_Time` bigint,"
"`2909_ΕΕΔΠ_Time` bigint,"
"`2910_ΑΕΔΠ_Time` bigint,"
"`2932_ΑνΕΔΠ_Time` bigint,"
"`Ekkr_ΕΔΠ_Time` bigint,"
"`2911_ΕΕΑΔΠ_Time` bigint,"
"`2912_ΑΕΑΔΠ_Time` bigint,"
"`Ekkr_ΕΑΔΠ_Time` bigint,"
"`2915_ΕΕΑΠ_Time` bigint,"
"`2916_ΑΕΑΠ_Time` bigint,"
"`2933_ΑνΕΑΠ_Time` bigint,"
"`Ekkr_ΕΑΠ_Time` bigint,"
"`2917_ΕΕΑΑΠ_Time` bigint,"
"`Ekkr_ΕΑΑΠ_Time` bigint,"
"`2921_ΕΕΚ_Time` bigint,"
"`2922_ΑΕΚ_Time` bigint,"
"`2934_ΑνΕΚ_Time` bigint,"
"`Ekkr_ΕΚ_Time` bigint,"
"`2923_ΕΕΑΚ_Time` bigint,"
"`2924_ΑΕΑΚ_Time` bigint,"
"`Ekkr_ΕΑΚ_Time` bigint,"
"InsertTimeStamp timestamp"																		
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"

delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date
#### type: ignore
)


##### Optimizer function

In [0]:
%python
# Helper function to create or replace table
def create_or_replace_table(table_name, df, partition_col):
    df.write.format("delta").mode("overwrite").partitionBy(partition_col).saveAsTable(table_name)
def create_or_replace_table_no_partition(table_name, df):
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
def repair_and_analyze_table(table_name):
    spark.sql(f"REPAIR TABLE {table_name} SYNC METADATA")
    spark.sql(f"ANALYZE TABLE {table_name} COMPUTE STATISTICS")    

##### MasterTable

In [0]:
mastertable = spark.sql(
    f'''
SELECT IFNULL(t.PartyTaxRegistrationNumber,t.CDI) ParentPartyId
	 , t.* 
     , CASE WHEN t.MenuCode IS NOT NULL THEN 1 else 0 END CMSFlag 
	 , CASE WHEN t.spv = 'GEMINI CORE SECURITISATION DAC' THEN 1 else 0 END GeMINiAccFlag
FROM `{env_prefix}-cdh-delivery`.repoint.core_mastertable t
WHERE t.BusinessDate = {unionRefDate!r}
''')
mastertable.createOrReplaceTempView("mastertable")

##### ModificationFullMapping

##### Eishghtika

In [0]:
Eishghtika_v1 = spark.sql(
f'''
		SELECT DISTINCT a.CDI, a.PartyTaxRegistrationNumber, a.ParentPartyId, a.AgreementKey, a.ExposureKey, a.AssetSegment, a.CMSFlag, a.ArrearsStatusENG, a.Balance, a.OnBalanceAmount, a.OffBalanceAmount, a.GeminiAccFlag, a.ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, ExternalCompany
		FROM `{env_prefix}-cdh-delivery`.repoint.int_corelegalaction a --interaction  a
		WHERE a.ActivityDate <= {run_date!r} AND a.ActivityDate > '2018-10-31' AND a.LegalActivityCategory = 'Εισηγητικά'
''')
Eishghtika_v1.createOrReplaceTempView("Eishghtika_v1")

In [0]:
Eishghtika = spark.sql(
f'''
select distinct * , rank() over(partition by A.ExposureKey,A.ActivityTypeKey order by A.ActivityDate desc, A.ExternalCompany asc ) Rank
from Eishghtika_v1 A
left anti join Eishghtika_v1 A1 ON A.ExposureKey=A1.ExposureKey AND A.ActivityTypeKey = '2925' AND A1.ActivityTypeKey not in ('4020','4021') AND IFNULL(A1.ActivityDate,'1900-01-01') <= IFNULL(a.ActivityDate,'1900-01-01')
left anti join Eishghtika_v1 A2 ON A.ExposureKey=A2.ExposureKey AND A.ActivityTypeKey = '1792' AND A2.ActivityTypeKey in ('2919','2920','2921','2922','2923','2924','2926','2929','2940','2951','1792','2934') AND IFNULL(A2.ActivityDate,'1900-01-01') <= IFNULL(A.ActivityDate,'1900-01-01')
left anti join Eishghtika_v1 A3 ON A.ExposureKey=A3.ExposureKey AND A.ActivityTypeKey = '1294' AND A3.ActivityTypeKey in ('2907','2908','2909','2910','2911','2912','2927','2950','1294','2932') AND IFNULL(A3.ActivityDate,'1900-01-01') <= IFNULL(A.ActivityDate,'1900-01-01')
left anti join Eishghtika_v1 A4 ON A.ExposureKey=A4.ExposureKey AND A.ActivityTypeKey = '2667' AND A4.ActivityTypeKey in ('4020','4021','4022','4023','4024','4025','2667') AND IFNULL(A4.ActivityDate,'1900-01-01') <= IFNULL(A.ActivityDate,'1900-01-01')
''')
Eishghtika.createOrReplaceTempView("Eishghtika")

##### Nomikes

In [0]:
Nomikes = spark.sql(
f'''
SELECT DISTINCT A.* , rank() over(partition by A.ExposureKey, A.LegalChapter order by A.ActivityDate asc, A.ExternalCompany asc, ActivityTypeKey desc) Rank 
FROM (
		SELECT DISTINCT a.CDI, a.PartyTaxRegistrationNumber, a.ParentPartyId, a.AgreementKey, a.ExposureKey, a.AssetSegment, a.CMSFlag, a.ArrearsStatusENG, a.Balance, ActivityDate, ActivityTypeKey, ActivityName, a.LegalActivityMapping, a.LegalChapter, a.ExternalCompany
		FROM  `{env_prefix}-cdh-delivery`.repoint.int_corelegalaction a --interaction a
		-- JOIN (SELECT DISTINCT ExposureKey FROM Eishghtika) e on e.ExposureKey = a.ExposureKey
		WHERE a.ActivityDate <= {run_date!r} AND a.ActivityDate > '2018-10-31' AND a.LegalActivityCategory = 'Νομικές Ενέργειες')A
INNER JOIN (SELECT ExposureKey, LegalActivityMapping, max(ActivityDate) ActivityDate
           FROM Eishghtika 
           WHERE LegalChapter = 'Έγκριση' or ActivityTypeKey in ('2926','2927','2929','2950','2951')
           GROUP BY ExposureKey, LegalActivityMapping) b ON A.ExposureKey = b.ExposureKey AND a.LegalActivityMapping = b.LegalActivityMapping
WHERE a.ActivityDate >= ifnull(b.ActivityDate,'1900-01-01')  
''')
Nomikes.createOrReplaceTempView("Nomikes") 

##### Nomikes_anag

In [0]:
Nomikes_anag_contract = spark.sql(
f'''
SELECT * 
FROM (
		SELECT DISTINCT  CDI, PartyTaxRegistrationNumber, ParentPartyId, AgreementKey, ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, ExternalCompany
		,rank() over(partition by AgreementKey, LegalChapter order by ActivityDate asc, ExternalCompany asc, ActivityTypeKey desc) Rank
		FROM Nomikes
		WHERE Rank = 1
		AND CMSFlag = 0) n
WHERE n.Rank = 1
''')
Nomikes_anag_contract.createOrReplaceTempView("Nomikes_anag_contract")   

In [0]:
Nomikes_anag_afm = spark.sql(
f'''
SELECT * 
FROM (
		SELECT DISTINCT  a.ParentPArtyId, a.ActivityDate, ActivityTypeKey, ActivityName, a.LegalActivityMapping, LegalChapter, ExternalCompany
		,rank() over(partition by a.ParentPArtyId, LegalChapter order by a.ActivityDate asc, ExternalCompany asc, ActivityTypeKey desc) Rank
		FROM Nomikes A
        LEFT JOIN (SELECT ParentPArtyId, LegalActivityMapping, max(ActivityDate) ActivityDate
                   FROM Eishghtika      
                   WHERE (LegalChapter = 'Έγκριση' or ActivityTypeKey in ('2926','2927','2929','2950','2951'))      
                   GROUP BY ParentPArtyId, LegalActivityMapping) b ON A.ParentPArtyId = b.ParentPArtyId AND a.LegalActivityMapping = b.LegalActivityMapping
        WHERE a.ActivityDate >= ifnull(b.ActivityDate,'1900-01-01')   
		  AND a.LegalActivityMapping = 'Κατάσχεση'  
		  AND Rank = 1
     ) a
WHERE a.Rank = 1
''')
Nomikes_anag_afm.createOrReplaceTempView("Nomikes_anag_afm")    

In [0]:
Nomikes_anag = spark.sql(
f'''
SELECT * 
FROM (
		SELECT ne.CDI, ne.PartyTaxRegistrationNumber, ne.ParentPartyId, ne.AgreementKey, acc.ExposureKey, ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, ExternalCompany
		FROM Nomikes_anag_contract ne
		JOIN mastertable acc on acc.AgreementKey = ne.AgreementKey
		WHERE LegalActivityMapping <> 'Κατάσχεση'
		UNION ALL
		SELECT acc.CDI, acc.PartyTaxRegistrationNumber, acc.ParentPartyId, acc.AgreementKey, acc.ExposureKey, ne.ActivityDate, ne.ActivityTypeKey, ne.ActivityName, ne.LegalActivityMapping, ne.LegalChapter, ne.ExternalCompany
		FROM Nomikes_anag_afm ne
		JOIN mastertable acc on acc.ParentPartyId = ne.ParentPartyId
		UNION ALL
		SELECT ne.CDI, ne.PartyTaxRegistrationNumber, ne.ParentPartyId, ne.AgreementKey, ne.ExposureKey, ne.ActivityDate, ne.ActivityTypeKey, ne.ActivityName, ne.LegalActivityMapping, ne.LegalChapter, ne.ExternalCompany
		FROM Nomikes ne
		WHERE CMSFlag = 1
		AND LegalActivityMapping <> 'Κατάσχεση') a
''')
Nomikes_anag.createOrReplaceTempView("Nomikes_anag")    

##### Eishghtika_anag

In [0]:
Eishghtika_anag_contract = spark.sql(
f'''
SELECT * 
FROM (
		SELECT DISTINCT e.CDI, e.PartyTaxRegistrationNumber, e.ParentPartyId, e.AgreementKey, e.ActivityDate, e.ActivityTypeKey, e.ActivityName, e.LegalActivityMapping,e.LegalChapter, e.ExternalCompany
		,rank() over(partition by e.AgreementKey,e.ActivityTypeKey order by e.ActivityDate desc, e.ExternalCompany asc) Rank
		FROM Eishghtika e
		LEFT JOIN (SELECT DISTINCT ParentPartyId, LegalActivityMapping FROM Nomikes_anag WHERE LegalActivityMapping = 'Κατάσχεση') n on e.ParentPartyId = n.ParentPartyId AND e.LegalActivityMapping = n.LegalActivityMapping 
		WHERE e.Rank = 1
		AND e.CMSFlag = 0
		AND n.ParentPartyId is null) e
WHERE e.Rank = 1
''')
Eishghtika_anag_contract.createOrReplaceTempView("Eishghtika_anag_contract")      

In [0]:
Eishghtika_anag_cms = spark.sql(
f'''
SELECT * 
FROM (
		SELECT e.CDI, e.PartyTaxRegistrationNumber, e.ParentPartyId, e.AgreementKey, e.ExposureKey, e.CMSFlag, e.ActivityDate, e.ActivityTypeKey, e.ActivityName, e.LegalActivityMapping, e.LegalChapter, e.ExternalCompany
		FROM Eishghtika e
		LEFT JOIN (SELECT DISTINCT ParentPartyId, LegalActivityMapping FROM Nomikes_anag WHERE LegalActivityMapping = 'Κατάσχεση') n on e.ParentPartyId = n.ParentPartyId AND e.LegalActivityMapping = n.LegalActivityMapping 
		WHERE e.Rank = 1
		AND e.CMSFlag = 1
		AND n.ParentPartyId is null) e
''')
Eishghtika_anag_cms.createOrReplaceTempView("Eishghtika_anag_cms")      

In [0]:
Eishghtika_anag_afm = spark.sql(
f'''
SELECT * 
FROM (
		SELECT DISTINCT e.PartyTaxRegistrationNumber, e.ParentPartyId, e.ActivityDate, e.ActivityTypeKey, e.ActivityName, e.LegalActivityMapping,e.LegalChapter, e.ExternalCompany
		,rank() over(partition by e.ParentPartyId,e.ActivityTypeKey order by e.ActivityDate desc, e.ExternalCompany asc) Rank
		FROM Eishghtika e
		inner JOIN (SELECT DISTINCT ParentPartyId, LegalActivityMapping FROM Nomikes_anag WHERE LegalActivityMapping = 'Κατάσχεση') n on e.ParentPartyId = n.ParentPartyId AND e.LegalActivityMapping = n.LegalActivityMapping 
		WHERE e.Rank = 1) e
WHERE e.Rank = 1
''')
Eishghtika_anag_afm.createOrReplaceTempView("Eishghtika_anag_afm")     

In [0]:
Eishghtika_anag = spark.sql(
f'''
SELECT * 
FROM (
		SELECT e.CDI, e.PartyTaxRegistrationNumber, e.ParentPartyId, e.AgreementKey, ExposureKey, CMSFlag, ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, ExternalCompany
		FROM Eishghtika_anag_contract e
		JOIN mastertable acc on acc.AgreementKey = e.AgreementKey
		UNION ALL
		SELECT CDI, e.PartyTaxRegistrationNumber, e.ParentPartyId, AgreementKey, ExposureKey, CMSFlag, ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, ExternalCompany
		FROM Eishghtika_anag_afm e
		JOIN mastertable acc on acc.ParentPartyId = e.ParentPartyId
		UNION ALL
		SELECT CDI, e.PartyTaxRegistrationNumber, e.ParentPartyId, AgreementKey, ExposureKey, CMSFlag, ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, ExternalCompany
		FROM Eishghtika_anag_cms e) a
''')
Eishghtika_anag.createOrReplaceTempView("Eishghtika_anag")  

create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_corereferrals", Eishghtika_anag)
repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_corereferrals")

In [0]:
# Perform the delete operation using SQL
spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A1
        WHERE A.ExposureKey = A1.ExposureKey
          AND A.LegalActivityMapping = A1.LegalActivityMapping
          AND A.LegalChapter IN ( 'Απόρριψη', 'Έγκριση', 'Αναμονή Έγκρισης')
          AND A1.LegalChapter IN ('Εισήγηση')
          AND IFNULL(A.ActivityDate, '1900-01-01') < IFNULL(A1.ActivityDate, '1900-01-01')
    )
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A1
        WHERE A.ExposureKey = A1.ExposureKey
          AND A.LegalActivityMapping = A1.LegalActivityMapping
          AND A.LegalChapter IN ('Απόρριψη', 'Έγκριση')
          AND A1.LegalChapter IN ('Αναμονή Έγκρισης')
          AND IFNULL(A.ActivityDate, '1900-01-01') < IFNULL(A1.ActivityDate, '1900-01-01')
    )
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A1
        WHERE A.ExposureKey = A1.ExposureKey
          AND A.LegalActivityMapping = A1.LegalActivityMapping
          AND A.LegalChapter IN ('Αναμονή Έγκρισης')
          AND A1.LegalChapter IN ('Απόρριψη', 'Έγκριση')
          AND IFNULL(A.ActivityDate, '1900-01-01') < IFNULL(A1.ActivityDate, '1900-01-01')
    )
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A1
        WHERE A.ExposureKey = A1.ExposureKey
          AND A.LegalActivityMapping = A1.LegalActivityMapping
          AND A.LegalChapter IN ('Έγκριση')
          AND A1.LegalChapter IN ('Απόρριψη')
          AND IFNULL(A.ActivityDate, '1900-01-01') <= IFNULL(A1.ActivityDate, '1900-01-01')
    )
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A1
        WHERE A.ExposureKey = A1.ExposureKey
          AND A.LegalActivityMapping = A1.LegalActivityMapping
          AND A.LegalChapter IN ('Απόρριψη')
          AND A1.LegalChapter IN ('Έγκριση')
          AND IFNULL(A.ActivityDate, '1900-01-01') < IFNULL(A1.ActivityDate, '1900-01-01')
    )
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals e_2911
        JOIN (SELECT ExposureKey, max(ActivityDate) ActivityDate FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals WHERE ActivityTypeKey in('2950','2927','2909') GROUP BY ExposureKey) e_2950 on e_2950.ExposureKey = e_2911.ExposureKey
        where e_2911.LegalActivityMapping = 'Διαταγή Πληρωμής' and e_2911.ActivityTypeKey = '2911' and IFNULL(e_2911.ActivityDate,'1900-01-01') >= IFNULL(e_2950.ActivityDate,'1900-01-01')
        and A.ExposureKey=e_2911.ExposureKey AND A.LegalActivityMapping = e_2911.LegalActivityMapping AND A.ActivityTypeKey <> e_2911.ActivityTypeKey
    )       
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals e_2923
        JOIN (SELECT AgreementKey, max(ActivityDate) ActivityDate FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals WHERE ActivityTypeKey in('2951','2926','2929','2921') GROUP BY AgreementKey) e_2951 on e_2951.AgreementKey = e_2923.AgreementKey
        where e_2923.LegalActivityMapping = 'Κατάσχεση' and e_2923.ActivityTypeKey = '2923' and IFNULL(e_2923.ActivityDate,'1900-01-01') >= IFNULL(e_2951.ActivityDate,'1900-01-01')
        and A.ExposureKey=e_2923.ExposureKey AND A.LegalActivityMapping = e_2923.LegalActivityMapping AND A.ActivityTypeKey <> e_2923.ActivityTypeKey
    )       
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals e_2935
        LEFT ANTI JOIN (SELECT DISTINCT ExposureKey, LegalActivityMapping FROM Nomikes_anag WHERE LegalActivityMapping = 'Κατάσχεση') n on n.ExposureKey = e_2935.ExposureKey
        where e_2935.LegalActivityMapping = 'Κατάσχεση' and e_2935.ActivityTypeKey = '2935'
        and A.ExposureKey=e_2935.ExposureKey AND A.LegalActivityMapping = e_2935.LegalActivityMapping  AND A.ActivityTypeKey in ('2907', '2908', '2909', '2910', '2911', '2912', '2927', '2932', '2950') AND IFNULL(A.ActivityDate,'1900-01-01') <= IFNULL(e_2935.ActivityDate,'1900-01-01')
    )       
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals e_4020
        LEFT ANTI JOIN `{env_prefix}-cdh-delivery`.repoint.int_corereferrals e_4021 on e_4021.ExposureKey = e_4021.ExposureKey AND e_4021.ActivityTypeKey in ('4021','4024','4025')
        where e_4020.ActivityTypeKey in ('4020','4022','4023') and A.ExposureKey=e_4020.ExposureKey AND A.LegalActivityMapping = e_4020.LegalActivityMapping AND A.ActivityTypeKey in ('2907', '2908', '2909', '2910', '2911', '2912', '2927', '2932', '2950') AND IFNULL(A.ActivityDate,'1900-01-01') <= IFNULL(e_4020.ActivityDate,'1900-01-01')
    )       
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A1
        WHERE A.ExposureKey=A1.ExposureKey AND A1.ActivityTypeKey = '2907' AND A.ActivityTypeKey = '2950' AND IFNULL(A.ActivityDate,'1900-01-01') <= IFNULL(A1.ActivityDate,'1900-01-01')
    )       
""")

spark.sql(f"""
    MERGE INTO `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    USING (
        SELECT e_2909.ExposureKey ExposureKey, e_2909.ActivityDate ActivityDate
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals e_2907 
        INNER JOIN `{env_prefix}-cdh-delivery`.repoint.int_corereferrals e_2909 
        ON e_2907.ExposureKey = e_2909.ExposureKey 
        WHERE e_2907.ActivityTypeKey = '2907' 
        AND e_2909.ActivityTypeKey = '2909'
    ) B 
    ON A.ExposureKey = B.ExposureKey 
    AND A.ActivityTypeKey = '2950' 
    WHEN MATCHED 
    AND A.ExposureKey = B.ExposureKey 
    AND A.ActivityTypeKey = '2950' 
    AND A.ActivityDate > B.ActivityDate 
    THEN 
    UPDATE SET A.ActivityDate = B.ActivityDate
""")

spark.sql(f"""
    MERGE INTO `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    USING (
        SELECT e_2921.ExposureKey ExposureKey, e_2921.ActivityDate ActivityDate
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals e_2919
        INNER JOIN `{env_prefix}-cdh-delivery`.repoint.int_corereferrals e_2921 
        ON e_2919.ExposureKey = e_2921.ExposureKey 
        WHERE e_2919.ActivityTypeKey = '2919' 
        AND e_2921.ActivityTypeKey = '2921'
    ) B 
    ON A.ExposureKey = B.ExposureKey 
    AND A.ActivityTypeKey = '2951' 
    WHEN MATCHED 
    AND A.ExposureKey = B.ExposureKey 
    AND A.ActivityTypeKey = '2951' 
    AND A.ActivityDate > B.ActivityDate 
    THEN 
    UPDATE SET A.ActivityDate = B.ActivityDate
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A1
        WHERE A.ExposureKey=A1.ExposureKey AND A1.ActivityTypeKey = '2909' AND A.ActivityTypeKey in('2927','2950') AND IFNULL(A.ActivityDate,'1900-01-01') <= IFNULL(A1.ActivityDate,'1900-01-01')
    )       
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A1
        WHERE A.ExposureKey=A1.ExposureKey AND A1.ActivityTypeKey in('2927','2950') AND A.ActivityTypeKey = '2909' AND IFNULL(A.ActivityDate,'1900-01-01') <= IFNULL(A1.ActivityDate,'1900-01-01')
    )       
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A1
        WHERE A.ExposureKey=A1.ExposureKey AND A1.ActivityTypeKey = '2919' AND A.ActivityTypeKey = '2951' AND IFNULL(A.ActivityDate,'1900-01-01') <= IFNULL(A1.ActivityDate,'1900-01-01')
    )       
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A1
        WHERE A.ExposureKey=A1.ExposureKey AND A1.ActivityTypeKey = '2921' AND A.ActivityTypeKey in('2926','2929','2951') AND IFNULL(A.ActivityDate,'1900-01-01') <= IFNULL(A1.ActivityDate,'1900-01-01')
    )       
""")

spark.sql(f"""
    DELETE FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A
    WHERE EXISTS (
        SELECT 1
        FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals A1
        WHERE A.ExposureKey=A1.ExposureKey AND A1.ActivityTypeKey in('2926','2929','2951') AND A.ActivityTypeKey = '2921' AND IFNULL(A.ActivityDate,'1900-01-01') <= IFNULL(A1.ActivityDate,'1900-01-01')
    )       
""")

##### Eishghtika_anag_egkr

In [0]:
Eishghtika_anag_egkr = spark.sql(
f'''
SELECT ExposureKey, LegalActivityMapping, max(ActivityDate) ActivityDate
FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals
WHERE LegalChapter = 'Έγκριση' or ActivityTypeKey in('2926','2927','2929','2950','2951')
GROUP BY ExposureKey, LegalActivityMapping
''')
Eishghtika_anag_egkr.createOrReplaceTempView("Eishghtika_anag_egkr")      

In [0]:
Nomikes_anag_0 = spark.sql(
f'''	
SELECT DISTINCT A.*
FROM Nomikes_anag A
LEFT JOIN (
		SELECT DISTINCT n.*
		FROM Nomikes_anag n
		LEFT JOIN Eishghtika_anag_egkr e on n.ExposureKey = e.ExposureKey AND n.LegalActivityMapping = e.LegalActivityMapping 
		WHERE IFNULL(n.ActivityDate,'1900-01-01') < IFNULL(e.ActivityDate,'9999-12-31')
)B ON A.ExposureKey=B.ExposureKey AND A.ActivityTypeKey=B.ActivityTypeKey AND A.ActivityDate=B.ActivityDate AND A.LegalActivityMapping=B.LegalActivityMapping
WHERE B.ExposureKey IS NULL AND B.ActivityTypeKey IS NULL AND B.ActivityDate IS NULL AND B.LegalActivityMapping IS NULL
''')
Nomikes_anag_0.createOrReplaceTempView("Nomikes_anag_0")   

##### ΑΓΩΓ_4022_4023

In [0]:
AGWG_4022_4023 = spark.sql(
f'''
SELECT * 
FROM (
		SELECT DISTINCT ExposureKey
		,concat(ActivityTypeKey,' - ',ActivityName) `ΑΓΩΓ_4022_4023`
		FROM (
		SELECT * , rank() over(partition by AgreementKey order by ActivityDate desc, ActivityTypeKey asc) Rank2
		FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals 
		WHERE ActivityTypeKey in ('4023','4022')
		AND CMSFlag = 0) a
		WHERE a.Rank2 = 1
		UNION ALL
		SELECT DISTINCT  ExposureKey
		,concat(ActivityTypeKey,' - ',ActivityName) `ΑΓΩΓ_4022_4023`
		FROM (
		SELECT * , rank() over(partition by ExposureKey order by ActivityDate desc, ActivityTypeKey asc) Rank2
		FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals
		WHERE ActivityTypeKey in ('4023','4022')
		AND CMSFlag = 1) a
		WHERE a.Rank2 = 1) b
''')
AGWG_4022_4023.createOrReplaceTempView("AGWG_4022_4023")      

##### Eishghtika_Perimeter_Contract

In [0]:
Eishghtika_Perimeter_Contract = spark.sql(
f'''
SELECT
    CDI,
    PartyTaxRegistrationNumber,
    ParentPartyId,
    AgreementKey,
    ExposureKey,
    `2901`    `2901_ΕΚΣ`,
    `2902`    `2902_ΕΑΚΣ`,
    `2903`    `2903_ΕΕΚΣ`,
    `2904`    `2904_ΑΕΚΣ`,
    `2905`    `2905_ΕΕΑΚΣ`,
    `2906`    `2906_ΑΕΑΚΣ`,
    `2907`    `2907_ΕΔΠ`,
    `2908`    `2908_ΕΑΔΠ`,
    `2909`    `2909_ΕΕΔΠ`,
    `2910`    `2910_ΑΕΔΠ`,
    `2911`    `2911_ΕΕΑΔΠ`,
    `2912`    `2912_ΑΕΑΔΠ`,
    `2913`    `2913_ΕΑΠ`,
    `2914`    `2914_ΕΑΑΠ`,
    `2915`    `2915_ΕΕΑΠ`,
    `2916`    `2916_ΑΕΑΠ`,
    `2917`    `2917_ΕΕΑΑΠ`,
    CASE WHEN IFNULL(`2950`, '1900-01-01') > IFNULL(`2927`, '1900-01-01') THEN `2950` ELSE `2927` END `2927_ΠΔΠ`,
    `2919`    `2919_ΕΚ`,
    `2920`    `2920_ΕΑΚ`,
    `2921`    `2921_ΕΕΚ`,
    CASE WHEN IFNULL(`2929`, '1900-01-01') > IFNULL(`2926`, '1900-01-01') AND IFNULL(`2929`, '1900-01-01') > IFNULL(`2951`, '1900-01-01') THEN `2929`
         WHEN IFNULL(`2951`, '1900-01-01') > IFNULL(`2926`, '1900-01-01') AND IFNULL(`2951`, '1900-01-01') > IFNULL(`2929`, '1900-01-01') THEN `2951`
         ELSE `2926` END `2926_ΠΚ`,
    `2922`    `2922_ΑΕΚ`,
    `2923`    `2923_ΕΕΑΚ`,
    `2924`    `2924_ΑΕΑΚ`,
    `2925`    `2925_ΑΕΝΕ`,
    `2940`    `2940_ΠΚΣ`,
    CASE WHEN IFNULL(`4022`, '1900-01-01') > IFNULL(`4020`, '1900-01-01') AND IFNULL(`4022`, '1900-01-01') > IFNULL(`4023`, '1900-01-01') THEN `4022`
         WHEN IFNULL(`4023`, '1900-01-01') > IFNULL(`4020`, '1900-01-01') AND IFNULL(`4023`, '1900-01-01') > IFNULL(`4022`, '1900-01-01') THEN `4023`
         ELSE `4020` END `4020_ΠΑΓ`,
    CASE WHEN IFNULL(`4024`, '1900-01-01') > IFNULL(`4021`, '1900-01-01') AND IFNULL(`4024`, '1900-01-01') > IFNULL(`4025`, '1900-01-01') THEN `4024`
         WHEN IFNULL(`4025`, '1900-01-01') > IFNULL(`4021`, '1900-01-01') AND IFNULL(`4025`, '1900-01-01') > IFNULL(`4024`, '1900-01-01') THEN `4025`
         ELSE `4021` END `4021_ΑΠΑΓ`,
    `2931`    `2931_ΑνΕΚΣ`,
    `2932`    `2932_ΑνΕΔΠ`,
    `2933`    `2933_ΑνΕΑΠ`,
    `2934`    `2934_ΑνΕΚ`,
    `2935`    `2935_EEKE`,
    CASE WHEN `2950` IS NOT NULL THEN '2950  -  Portfolio Management ΔΠ' END `ΠΔ_2950`,
    CASE WHEN `2951` IS NOT NULL THEN '2951  -  Portfolio Management ΚΤΧ' END `ΚΣΧ_2951`
FROM (
    SELECT  
        ExposureKey,
        CDI,
        PartyTaxRegistrationNumber,
        ParentPartyId,
        AgreementKey,
        ActivityTypeKey,
        ActivityDate
    FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals
) as TableToPivot
PIVOT (
    MIN(ActivityDate) FOR ActivityTypeKey IN (
        "2901", "2902", "2903", "2904", "2905", "2906", "2907", "2908", "2909", "2910", "2911", "2912", "2913", "2914", "2915", "2916", "2917", "2927", "2919", "2920", "2921", "2926", "2929", "2922", "2923", "2924", "2925", "2940", "4020", "4022", "4023", "4021", "4024", "4025", "2931", "2932", "2933", "2934", "2935", "2950", "2951")
)
''')
Eishghtika_Perimeter_Contract.createOrReplaceTempView("Eishghtika_Perimeter_Contract")

##### Eishghtika_Agency

In [0]:
Eishghtika_Agency = spark.sql(f'''
SELECT AgreementKey
		,ExposureKey
		,`2901`			`AG_ΕΚΣ`
		,`2907`			`AG_ΕΔΠ`
		,`2913`			`AG_ΕΑΠ`
		,`2919`			`AG_ΕΚ`
		,`4020`			`AG_ΑΓΩΓ`
		,IFNULL(`2909`,`2927`) `AG_ΔΠ`
		,IFNULL(`2909`,`2927`) `AG_ΚΤΣΧ`
FROM (
		SELECT  AgreementKey
		,ExposureKey
		,ActivityTypeKey
		,ExternalCompany
		FROM `{env_prefix}-cdh-delivery`.repoint.int_corereferrals
		) as TableToPivot
pivot(min(ExternalCompany) for ActivityTypeKey in("2901","2907","2913","2919","4020","2909","2927"))                                          
''')
Eishghtika_Agency.createOrReplaceTempView("Eishghtika_Agency")

##### Nomikes_Perimeter_Contract

In [0]:
Nomikes_Perimeter_Contract = spark.sql(
    '''
    SELECT
        AgreementKey,
        ExposureKey,
        MIN(CASE WHEN LegalChapter = 'ΚΑΤΑ_ΔΠ' THEN ActivityDate END) AS `ΚΑΤΑ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΕΚΔ_ΔΠ' THEN ActivityDate END) AS `ΕΚΔ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΕΠΙ_ΔΠ' THEN ActivityDate END) AS `ΕΠΙ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΚΑΤΑΣΧ' THEN ActivityDate END) AS `ΚΑΤΑΣΧ`,
        MIN(CASE WHEN LegalChapter = 'ΚΑΤΑ_ΑΓΩΓ' THEN ActivityDate END) AS `ΚΑΤΑ_ΑΓΩΓ`,
        MIN(CASE WHEN LegalChapter = 'ΕΚΔ_ΑΓΩΓ' THEN ActivityDate END) AS `ΕΚΔ_ΑΓΩΓ`,
        MIN(CASE WHEN LegalChapter = 'ΚΟΙΝ_ΑΓΩΓ' THEN ActivityDate END) AS `ΚΟΙΝ_ΑΓΩΓ`,
        MIN(CASE WHEN LegalChapter = 'ΠΡΟΣΗΜ' THEN ActivityDate END) AS `ΠΡΟΣΗΜ`,
        MIN(CASE WHEN LegalChapter = 'ΚΑΤΑ_ΔΠ' THEN ExternalCompany END) AS `AG_ΚΑΤΑ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΕΚΔ_ΔΠ' THEN ExternalCompany END) AS `AG_ΕΚΔ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΕΠΙ_ΔΠ' THEN ExternalCompany END) AS `AG_ΕΠΙ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΚΑΤΑΣΧ' THEN ExternalCompany END) AS `AG_ΚΑΤΑΣΧ`
    FROM Nomikes_anag_0
    GROUP BY AgreementKey, ExposureKey
    '''
)
Nomikes_Perimeter_Contract.createOrReplaceTempView("Nomikes_Perimeter_Contract")

##### legalprocess

In [0]:
%python
LegalProcess = spark.sql(f'''
SELECT *
FROM `{env_prefix}-cdh-delivery`.repoint.core_legalmodule
WHERE BusinessDate={run_date!r}
''')
LegalProcess.createOrReplaceTempView("LegalProcess")

##### Referrals 

In [0]:
%python
referrals = spark.sql(f'''
SELECT
		{run_date!r} as BusinessDate
		,ep.CDI
		,ep.PartyTaxRegistrationNumber
		,ep.ParentPartyId
		,ep.AgreementKey
		,acc.ExposureKey
		,acc.GeminiAccFlag
		,acc.AccountStatus as ArrearsStatusENG
		,acc.AssetSegment
		,acc.Balance
		,acc.OnBalanceAmount
		,acc.OffBalanceAmount
		,dl.AGENCY_NAME AgencyCode
        -- ,cca.Agency AgencyCode
		,`2901_ΕΚΣ`
		,`2902_ΕΑΚΣ`
		,`2903_ΕΕΚΣ`
		,`2904_ΑΕΚΣ`
		,`2905_ΕΕΑΚΣ`
		,`2906_ΑΕΑΚΣ`
		,`2907_ΕΔΠ`
		,`2908_ΕΑΔΠ`
		,`2909_ΕΕΔΠ`
		,`2910_ΑΕΔΠ`
		,`2911_ΕΕΑΔΠ`
		,`2912_ΑΕΑΔΠ`
		,`2913_ΕΑΠ`
		,`2914_ΕΑΑΠ`
		,`2915_ΕΕΑΠ`
		,`2916_ΑΕΑΠ`
		,`2917_ΕΕΑΑΠ`
		,`2927_ΠΔΠ`
		,`2919_ΕΚ`
		,`2920_ΕΑΚ`
		,`2921_ΕΕΚ`
		,`2926_ΠΚ`
		,`2922_ΑΕΚ`
		,`2923_ΕΕΑΚ`
		,`2924_ΑΕΑΚ`
		,`2925_ΑΕΝΕ`
		,`2940_ΠΚΣ`
		,`4020_ΠΑΓ`
		,`4021_ΑΠΑΓ`
		,`2931_ΑνΕΚΣ`
		,`2932_ΑνΕΔΠ`
		,`2933_ΑνΕΑΠ`
		,`2934_ΑνΕΚ`
		,`2935_EEKE`
		,IF(`2909_ΕΕΔΠ` IS NOT NULL, `2909_ΕΕΔΠ`,`2927_ΠΔΠ`) `EDP_Date`
		,IF(`2921_ΕΕΚ` IS NOT NULL, `2921_ΕΕΚ`,`2926_ΠΚ`) `EK_Date`
		,`ΠΔ_2950`
		,`ΚΣΧ_2951`
		,`AG_ΕΚΣ`
		,`AG_ΕΔΠ`
		,`AG_ΕΑΠ`
		,`AG_ΕΚ`
		,`ΚΑΤΑ_ΔΠ`
		,`ΕΚΔ_ΔΠ`
		,`ΕΠΙ_ΔΠ`
		,`ΚΑΤΑΣΧ` 
		,`ΚΑΤΑ_ΑΓΩΓ`
		,`ΕΚΔ_ΑΓΩΓ`
		,`ΚΟΙΝ_ΑΓΩΓ`
		,`ΠΡΟΣΗΜ`  
		,`AG_ΚΑΤΑ_ΔΠ`
		,`AG_ΕΚΔ_ΔΠ`
		,`AG_ΕΠΙ_ΔΠ`
		,`AG_ΚΑΤΑΣΧ`
		,`ΑΓΩΓ_4022_4023`
		,CASE WHEN ep.`4020_ΠΑΓ` IS NOT NULL AND ep.`4021_ΑΠΑΓ` IS NULL AND `ΚΟΙΝ_ΑΓΩΓ` IS NOT NULL  then 'ΚΟΙΝ_ΑΓΩΓ'
		 	  WHEN ep.`4020_ΠΑΓ` IS NOT NULL AND ep.`4021_ΑΠΑΓ` IS NULL AND `ΕΚΔ_ΑΓΩΓ` IS NOT NULL   then 'ΕΚΔ_ΑΓΩΓ'
		 	  WHEN ep.`4020_ΠΑΓ` IS NOT NULL AND ep.`4021_ΑΠΑΓ` IS NULL AND `ΚΑΤΑ_ΑΓΩΓ` IS NOT NULL  then 'ΚΑΤΑ_ΑΓΩΓ'
		 	  WHEN ep.`4020_ΠΑΓ` IS NOT NULL AND ep.`4021_ΑΠΑΓ` IS NULL AND lp.`CurrentAgency_agog` IS NOT NULL AND lp.`AssignmentDate_agog` IS NOT NULL then 'ΑΝΕΥ Ν.Ε.'
		 	  WHEN ep.`4020_ΠΑΓ` IS NOT NULL AND ep.`4021_ΑΠΑΓ` IS NULL AND (lp.`CurrentAgency_agog` IS NULL OR lp.`AssignmentDate_agog` IS NULL) then 'Προς ανάθεση'
		  END `ΑΓΩΓ_Category` 
		,CASE WHEN (ep.`2909_ΕΕΔΠ` IS NOT NULL OR `2927_ΠΔΠ` IS NOT NULL) AND  `ΕΠΙ_ΔΠ` IS NOT NULL    then 'ΕΠΙ_ΔΠ'
		 	  WHEN (ep.`2909_ΕΕΔΠ` IS NOT NULL OR `2927_ΠΔΠ` IS NOT NULL) AND  `ΕΚΔ_ΔΠ` IS NOT NULL    then 'ΕΚΔ_ΔΠ'
		 	  WHEN (ep.`2909_ΕΕΔΠ` IS NOT NULL OR `2927_ΠΔΠ` IS NOT NULL) AND  `ΚΑΤΑ_ΔΠ` IS NOT NULL   then 'ΚΑΤΑ_ΔΠ'
		 	  WHEN (ep.`2909_ΕΕΔΠ` IS NOT NULL OR `2927_ΠΔΠ` IS NOT NULL) AND lp.`CurrentAgency_dp` IS NOT NULL AND lp.`AssignmentDate_dp` IS NOT NULL then 'ΑΝΕΥ Ν.Ε.'
		 	  WHEN (ep.`2909_ΕΕΔΠ` IS NOT NULL OR `2927_ΠΔΠ` IS NOT NULL) AND (lp.`CurrentAgency_dp` IS NULL OR lp.`AssignmentDate_dp` IS NULL) then 'Προς ανάθεση'
		 END `ΔΠ_Category`
		,CASE WHEN (ep.`2921_ΕΕΚ` IS NOT NULL OR `2926_ΠΚ` IS NOT NULL OR `2935_EEKE` IS NOT NULL) AND  `ΚΑΤΑΣΧ` IS NOT NULL    then 'ΚΑΤΑΣΧΕΣΗ'
		 	  WHEN (ep.`2921_ΕΕΚ` IS NOT NULL OR `2926_ΠΚ` IS NOT NULL OR `2935_EEKE` IS NOT NULL) AND lp.`CurrentAgency_ktsx` IS NOT NULL AND lp.`AssignmentDate_ktsx` IS NOT NULL then 'ΑΝΕΥ Ν.Ε.'
		 	  WHEN (ep.`2921_ΕΕΚ` IS NOT NULL OR `2926_ΠΚ` IS NOT NULL OR `2935_EEKE` IS NOT NULL) AND (lp.`CurrentAgency_ktsx` IS NULL OR lp.`AssignmentDate_ktsx` IS NULL) then 'Προς ανάθεση'
		  END `ΚΤΣΧ_Category`
		,CASE WHEN ep.`2915_ΕΕΑΠ` IS NOT NULL AND `ΠΡΟΣΗΜ` IS NOT NULL    then 'ΠΡΟΣΗΜΕΙΩΣΗ'
		 	   WHEN ep.`2915_ΕΕΑΠ` IS NOT NULL AND lp.`CurrentAgency_pros` IS NOT NULL AND lp.`AssignmentDate_pros` IS NOT NULL then 'ΑΝΕΥ Ν.Ε.'
		 	   WHEN ep.`2915_ΕΕΑΠ` IS NOT NULL AND (lp.`CurrentAgency_pros` IS NULL OR lp.`AssignmentDate_pros` IS NULL) then 'Προς ανάθεση'
		  END `ΠΡΟΣΗΜ_Category`
		,CASE 
			  WHEN `2909_ΕΕΔΠ` IS NOT NULL OR `2927_ΠΔΠ` IS NOT NULL THEN 'Έγκριση'
			  WHEN `2910_ΑΕΔΠ` IS NOT NULL THEN 'Απόρριψη'
			  WHEN `2907_ΕΔΠ`  IS NOT NULL THEN 'Έισήγηση'
			  END AS `ΔΠ_Status`
		,CASE 
			  WHEN `2909_ΕΕΔΠ` IS NOT NULL OR `2927_ΠΔΠ` IS NOT NULL THEN IFNULL(`2909_ΕΕΔΠ`, `2927_ΠΔΠ`) 
			  WHEN `2910_ΑΕΔΠ` IS NOT NULL THEN `2910_ΑΕΔΠ`
			  WHEN `2907_ΕΔΠ`  IS NOT NULL THEN `2907_ΕΔΠ` 
			  END AS `ΔΠ_Status_Date`
		,CASE WHEN ep.`4020_ΠΑΓ` IS NOT NULL AND ep.`4021_ΑΠΑΓ` IS NULL THEN 'Eligible'
			  WHEN ep.`4021_ΑΠΑΓ` IS NOT NULL							THEN 'Not Eligible'
			  END AS `AgogiStatus`
		,CASE WHEN ep.`4020_ΠΑΓ` IS NOT NULL AND ep.`4021_ΑΠΑΓ` IS NULL THEN `4020_ΠΑΓ`
			  WHEN ep.`4021_ΑΠΑΓ` IS NOT NULL							THEN `4021_ΑΠΑΓ`
			  END AS `Agogi_Status_Date`
		,CASE 
			  WHEN `2921_ΕΕΚ` IS NOT NULL OR `2926_ΠΚ` IS NOT NULL OR `2940_ΠΚΣ` IS NOT NULL THEN 'Έγκριση'
			  WHEN `2922_ΑΕΚ` IS NOT NULL THEN 'Απόρριψη'
			  WHEN `2919_ΕΚ`  IS NOT NULL THEN 'Έισήγηση'
			  END AS `KSX_Status`
		,CASE 
			  WHEN `2921_ΕΕΚ` IS NOT NULL OR `2926_ΠΚ` IS NOT NULL OR `2940_ΠΚΣ` IS NOT NULL THEN IFNULL(IFNULL(`2921_ΕΕΚ`, `2926_ΠΚ`),`2940_ΠΚΣ`)
			  WHEN `2922_ΑΕΚ` IS NOT NULL THEN `2922_ΑΕΚ`
			  WHEN `2919_ΕΚ`  IS NOT NULL THEN `2919_ΕΚ`
			  END AS `KSX_Status_Date`
		,lp.`AssignmentDate_agog` AS `entoli_TA`
		,lp.`AssignmentDate_dp` AS  `entoli_dp`
		,lp.`AssignmentDate_ktsx` AS `entoli_ksx`
		,lp.`AssignmentDate_pros` AS `entoli_pros`
		,lp.`CurrentAgency_agog` AS `ΑΓΩΓ_CurrentAgency`
		,lp.`CurrentAgency_dp` AS `ΔΠ_CurrentAgency`
		,lp.`CurrentAgency_ktsx` AS `ΚΤΣΧ_CurrentAgency`
		,lp.`CurrentAgency_pros` AS `ΠΡΟΣ_CurrentAgency`		
		, -datediff(DAY,ep.`2903_ΕΕΚΣ` ,ep.`2901_ΕΚΣ`) `2903_ΕΕΚΣ_Time`
	  , -datediff(DAY,ep.`2904_ΑΕΚΣ` ,ep.`2901_ΕΚΣ`) `2904_ΑΕΚΣ_Time`
		, -datediff(DAY,ep.`2931_ΑνΕΚΣ` ,ep.`2901_ΕΚΣ`) `2931_ΑνΕΚΣ_Time`
		, -datediff(DAY,{run_date!r}, ep.`2901_ΕΚΣ`) `Ekkr_ΕΚΣ_Time`
		, -datediff(DAY,ep.`2905_ΕΕΑΚΣ` ,ep.`2902_ΕΑΚΣ`) `2905_ΕΕΑΚΣ_Time`
		, -datediff(DAY,ep.`2906_ΑΕΑΚΣ` ,ep.`2902_ΕΑΚΣ`) `2906_ΑΕΑΚΣ_Time`
		, -datediff(DAY,{run_date!r} ,ep.`2902_ΕΑΚΣ`) `Ekkr_ΕΑΚΣ_Time`
		, -datediff(DAY,ep.`2909_ΕΕΔΠ` ,ep.`2907_ΕΔΠ`) `2909_ΕΕΔΠ_Time`
		, -datediff(DAY,ep.`2910_ΑΕΔΠ` ,ep.`2907_ΕΔΠ`) `2910_ΑΕΔΠ_Time`
		, -datediff(DAY,ep.`2932_ΑνΕΔΠ` ,ep.`2907_ΕΔΠ`) `2932_ΑνΕΔΠ_Time`
		, -datediff(DAY,{run_date!r} ,ep.`2907_ΕΔΠ`) `Ekkr_ΕΔΠ_Time`
		, -datediff(DAY,ep.`2911_ΕΕΑΔΠ` ,ep.`2908_ΕΑΔΠ`) `2911_ΕΕΑΔΠ_Time`
		, -datediff(DAY,ep.`2912_ΑΕΑΔΠ` ,ep.`2908_ΕΑΔΠ`) `2912_ΑΕΑΔΠ_Time`
		, -datediff(DAY,{run_date!r} ,ep.`2908_ΕΑΔΠ`) `Ekkr_ΕΑΔΠ_Time`
		, -datediff(DAY,ep.`2915_ΕΕΑΠ` ,ep.`2913_ΕΑΠ`) `2915_ΕΕΑΠ_Time`
		, -datediff(DAY,ep.`2916_ΑΕΑΠ` ,ep.`2913_ΕΑΠ`) `2916_ΑΕΑΠ_Time`
		, -datediff(DAY,ep.`2933_ΑνΕΑΠ` ,ep.`2913_ΕΑΠ`) `2933_ΑνΕΑΠ_Time`
		, -datediff(DAY,{run_date!r} ,ep.`2913_ΕΑΠ`) `Ekkr_ΕΑΠ_Time`
		, -datediff(DAY,ep.`2917_ΕΕΑΑΠ` ,ep.`2914_ΕΑΑΠ`) `2917_ΕΕΑΑΠ_Time`
		, -datediff(DAY,{run_date!r} ,ep.`2914_ΕΑΑΠ`) `Ekkr_ΕΑΑΠ_Time`
		, -datediff(DAY,ep.`2921_ΕΕΚ` ,ep.`2919_ΕΚ`) `2921_ΕΕΚ_Time`
		, -datediff(DAY,ep.`2922_ΑΕΚ` ,ep.`2919_ΕΚ`) `2922_ΑΕΚ_Time`
		, -datediff(DAY,ep.`2934_ΑνΕΚ` ,ep.`2919_ΕΚ`) `2934_ΑνΕΚ_Time`
		, -datediff(DAY,{run_date!r} ,ep.`2919_ΕΚ`) `Ekkr_ΕΚ_Time`
		, -datediff(DAY,ep.`2923_ΕΕΑΚ` ,ep.`2920_ΕΑΚ`) `2923_ΕΕΑΚ_Time`
		, -datediff(DAY,ep.`2924_ΑΕΑΚ` ,ep.`2920_ΕΑΚ`) `2924_ΑΕΑΚ_Time`
		, -datediff(DAY,{run_date!r} ,ep.`2920_ΕΑΚ`) `Ekkr_ΕΑΚ_Time`		
FROM Eishghtika_Perimeter_Contract ep
INNER JOIN mastertable acc on ep.ExposureKey = acc.ExposureKey and ep.ParentPartyId = acc.ParentPartyId
LEFT JOIN `{env_prefix}-cdh`.landed.repointqcab_daily_portfolio dl on acc.ExposureKey = dl.ACCOUNT_NUMBER AND acc.CDI = dl.CUST_CIF AND dl.BusinessDate={run_date!r}  
LEFT JOIN Eishghtika_Agency eg on ep.ExposureKey = eg.ExposureKey
LEFT JOIN Nomikes_Perimeter_Contract np on np.ExposureKey = ep.ExposureKey 
LEFT JOIN AGWG_4022_4023 ag on ag.ExposureKey = ep.ExposureKey
LEFT JOIN LegalProcess lp on lp.AgreementKey = ep.AgreementKey
''')
referrals.createOrReplaceTempView("referrals")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 
  cast(BusinessDate as string) as BusinessDate
, cast(CDI as string) as CDI
, cast(PartyTaxRegistrationNumber as string) as PartyTaxRegistrationNumber
, cast(ParentPartyId as string) as ParentPartyId
, cast(AgreementKey as string) as AgreementKey
, cast(ExposureKey as string) as ExposureKey
, cast(GeminiAccFlag as int) as GeminiAccFlag
, cast(ArrearsStatusENG as string) as ArrearsStatusENG
, cast(AssetSegment as string) as AssetSegment
, cast(Balance as string) as Balance
, cast(OnBalanceAmount as string) as OnBalanceAmount
, cast(OffBalanceAmount as string) as OffBalanceAmount
, cast(AgencyCode as string) as AgencyCode
, cast(`2901_ΕΚΣ` as date) as `2901_ΕΚΣ`
, cast(`2902_ΕΑΚΣ` as date) as `2902_ΕΑΚΣ`
, cast(`2903_ΕΕΚΣ` as date) as `2903_ΕΕΚΣ`
, cast(`2904_ΑΕΚΣ` as date) as `2904_ΑΕΚΣ`
, cast(`2905_ΕΕΑΚΣ` as date) as `2905_ΕΕΑΚΣ`
, cast(`2906_ΑΕΑΚΣ` as date) as `2906_ΑΕΑΚΣ`
, cast(`2907_ΕΔΠ` as date) as `2907_ΕΔΠ`
, cast(`2908_ΕΑΔΠ` as date) as `2908_ΕΑΔΠ`
, cast(`2909_ΕΕΔΠ` as date) as `2909_ΕΕΔΠ`
, cast(`2910_ΑΕΔΠ` as date) as `2910_ΑΕΔΠ`
, cast(`2911_ΕΕΑΔΠ` as date) as `2911_ΕΕΑΔΠ`
, cast(`2912_ΑΕΑΔΠ` as date) as `2912_ΑΕΑΔΠ`
, cast(`2913_ΕΑΠ` as date) as `2913_ΕΑΠ`
, cast(`2914_ΕΑΑΠ` as date) as `2914_ΕΑΑΠ`
, cast(`2915_ΕΕΑΠ` as date) as `2915_ΕΕΑΠ`
, cast(`2916_ΑΕΑΠ` as date) as `2916_ΑΕΑΠ`
, cast(`2917_ΕΕΑΑΠ` as date) as `2917_ΕΕΑΑΠ`
, cast(`2927_ΠΔΠ` as date) as `2927_ΠΔΠ`
, cast(`2919_ΕΚ` as date) as `2919_ΕΚ`
, cast(`2920_ΕΑΚ` as date) as `2920_ΕΑΚ`
, cast(`2921_ΕΕΚ` as date) as `2921_ΕΕΚ`
, cast(`2926_ΠΚ` as date) as `2926_ΠΚ`
, cast(`2922_ΑΕΚ` as date) as `2922_ΑΕΚ`
, cast(`2923_ΕΕΑΚ` as date) as `2923_ΕΕΑΚ`
, cast(`2924_ΑΕΑΚ` as date) as `2924_ΑΕΑΚ`
, cast(`2925_ΑΕΝΕ` as date) as `2925_ΑΕΝΕ`
, cast(`2940_ΠΚΣ` as date) as `2940_ΠΚΣ`
, cast(`4020_ΠΑΓ` as date) as `4020_ΠΑΓ`
, cast(`4021_ΑΠΑΓ` as date) as `4021_ΑΠΑΓ`
, cast(`2931_ΑνΕΚΣ` as date) as `2931_ΑνΕΚΣ`
, cast(`2932_ΑνΕΔΠ` as date) as `2932_ΑνΕΔΠ`
, cast(`2933_ΑνΕΑΠ` as date) as `2933_ΑνΕΑΠ`
, cast(`2934_ΑνΕΚ` as date) as `2934_ΑνΕΚ`
, cast(`2935_EEKE` as date) as `2935_EEKE`
, cast(`EDP_Date` as date) as `EDP_Date`
, cast(`EK_Date` as date) as `EK_Date`
, cast(`ΠΔ_2950` as string) as `ΠΔ_2950`
, cast(`ΚΣΧ_2951` as string) as `ΚΣΧ_2951`
, cast(`AG_ΕΚΣ` as string) as `AG_ΕΚΣ`
, cast(`AG_ΕΔΠ` as string) as `AG_ΕΔΠ`
, cast(`AG_ΕΑΠ` as string) as `AG_ΕΑΠ`
, cast(`AG_ΕΚ` as string) as `AG_ΕΚ`
, cast(`ΚΑΤΑ_ΔΠ` as date) as `ΚΑΤΑ_ΔΠ`
, cast(`ΕΚΔ_ΔΠ` as date) as `ΕΚΔ_ΔΠ`
, cast(`ΕΠΙ_ΔΠ` as date) as `ΕΠΙ_ΔΠ`
, cast(`ΚΑΤΑΣΧ` as date) as `ΚΑΤΑΣΧ`
, cast(`ΚΑΤΑ_ΑΓΩΓ` as date) as `ΚΑΤΑ_ΑΓΩΓ`
, cast(`ΕΚΔ_ΑΓΩΓ` as date) as `ΕΚΔ_ΑΓΩΓ`
, cast(`ΚΟΙΝ_ΑΓΩΓ` as date) as `ΚΟΙΝ_ΑΓΩΓ`
, cast(`ΠΡΟΣΗΜ` as date) as `ΠΡΟΣΗΜ`
, cast(`AG_ΚΑΤΑ_ΔΠ` as string) as `AG_ΚΑΤΑ_ΔΠ`
, cast(`AG_ΕΚΔ_ΔΠ` as string) as `AG_ΕΚΔ_ΔΠ`
, cast(`AG_ΕΠΙ_ΔΠ` as string) as `AG_ΕΠΙ_ΔΠ`
, cast(`AG_ΚΑΤΑΣΧ` as string) as `AG_ΚΑΤΑΣΧ`
, cast(`ΑΓΩΓ_4022_4023` as string) as `ΑΓΩΓ_4022_4023`
, cast(`ΑΓΩΓ_Category` as string) as `ΑΓΩΓ_Category`
, cast(`ΔΠ_Category` as string) as `ΔΠ_Category`
, cast(`ΚΤΣΧ_Category` as string) as `ΚΤΣΧ_Category`
, cast(`ΠΡΟΣΗΜ_Category` as string) as `ΠΡΟΣΗΜ_Category`
, cast(`ΔΠ_Status` as string) as `ΔΠ_Status`
, cast(`ΔΠ_Status_Date` as date) as `ΔΠ_Status_Date`
, cast(`AgogiStatus` as string) as `AgogiStatus`
, cast(`Agogi_Status_Date` as date) as `Agogi_Status_Date`
, cast(`KSX_Status` as string) as `KSX_Status`
, cast(`KSX_Status_Date` as date) as `KSX_Status_Date`
, cast(`entoli_TA` as date) as `entoli_TA`
, cast(`entoli_dp` as date) as `entoli_dp`
, cast(`entoli_ksx` as date) as `entoli_ksx`
, cast(`entoli_pros` as date) as `entoli_pros`
, cast(`ΑΓΩΓ_CurrentAgency` as string) as `ΑΓΩΓ_CurrentAgency`
, cast(`ΔΠ_CurrentAgency` as string) as `ΔΠ_CurrentAgency`
, cast(`ΚΤΣΧ_CurrentAgency` as string) as `ΚΤΣΧ_CurrentAgency`
, cast(`ΠΡΟΣ_CurrentAgency` as string) as `ΠΡΟΣ_CurrentAgency`
, cast(`2903_ΕΕΚΣ_Time` as bigint) as `2903_ΕΕΚΣ_Time`
, cast(`2904_ΑΕΚΣ_Time` as bigint) as `2904_ΑΕΚΣ_Time`
, cast(`2931_ΑνΕΚΣ_Time` as bigint) as `2931_ΑνΕΚΣ_Time`
, cast(`Ekkr_ΕΚΣ_Time` as bigint) as `Ekkr_ΕΚΣ_Time`
, cast(`2905_ΕΕΑΚΣ_Time` as bigint) as `2905_ΕΕΑΚΣ_Time`
, cast(`2906_ΑΕΑΚΣ_Time` as bigint) as `2906_ΑΕΑΚΣ_Time`
, cast(`Ekkr_ΕΑΚΣ_Time` as bigint) as `Ekkr_ΕΑΚΣ_Time`
, cast(`2909_ΕΕΔΠ_Time` as bigint) as `2909_ΕΕΔΠ_Time`
, cast(`2910_ΑΕΔΠ_Time` as bigint) as `2910_ΑΕΔΠ_Time`
, cast(`2932_ΑνΕΔΠ_Time` as bigint) as `2932_ΑνΕΔΠ_Time`
, cast(`Ekkr_ΕΔΠ_Time` as bigint) as `Ekkr_ΕΔΠ_Time`
, cast(`2911_ΕΕΑΔΠ_Time` as bigint) as `2911_ΕΕΑΔΠ_Time`
, cast(`2912_ΑΕΑΔΠ_Time` as bigint) as `2912_ΑΕΑΔΠ_Time`
, cast(`Ekkr_ΕΑΔΠ_Time` as bigint) as `Ekkr_ΕΑΔΠ_Time`
, cast(`2915_ΕΕΑΠ_Time` as bigint) as `2915_ΕΕΑΠ_Time`
, cast(`2916_ΑΕΑΠ_Time` as bigint) as `2916_ΑΕΑΠ_Time`
, cast(`2933_ΑνΕΑΠ_Time` as bigint) as `2933_ΑνΕΑΠ_Time`
, cast(`Ekkr_ΕΑΠ_Time` as bigint) as `Ekkr_ΕΑΠ_Time`
, cast(`2917_ΕΕΑΑΠ_Time` as bigint) as `2917_ΕΕΑΑΠ_Time`
, cast(`Ekkr_ΕΑΑΠ_Time` as bigint) as `Ekkr_ΕΑΑΠ_Time`
, cast(`2921_ΕΕΚ_Time` as bigint) as `2921_ΕΕΚ_Time`
, cast(`2922_ΑΕΚ_Time` as bigint) as `2922_ΑΕΚ_Time`
, cast(`2934_ΑνΕΚ_Time` as bigint) as `2934_ΑνΕΚ_Time`
, cast(`Ekkr_ΕΚ_Time` as bigint) as `Ekkr_ΕΚ_Time`
, cast(`2923_ΕΕΑΚ_Time` as bigint) as `2923_ΕΕΑΚ_Time`
, cast(`2924_ΑΕΑΚ_Time` as bigint) as `2924_ΑΕΑΚ_Time`
, cast(`Ekkr_ΕΑΚ_Time` as bigint) as `Ekkr_ΕΑΚ_Time`
, cast(getdate() as timestamp) as InsertTimeStamp
from referrals
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_Referrals: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
Referrals = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
Referrals.createOrReplaceTempView("Referrals")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/Referrals/"
file_name = "Referrals_" +run_date+ ".csv"
Referrals.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)